# 04 — PySpark Preprocessing
**Projet** : PFE Scoring Crédit Bancaire  
**Dataset** : `marts_marts.dataset_ml_v3` — 8 570 101 lignes | 105 colonnes  
**Objectif** : Préparer le dataset pour l'entraînement des modèles ML

---
### Pipeline
1. Configuration PySpark + Connexion PostgreSQL
2. Chargement du dataset
3. Split Temporel (Train / Validation / Test / Scoring)
4. Exploration et Analyse des NULLs
5. Imputation des valeurs manquantes
6. Encodage des variables catégorielles
7. Analyse de redondance entre colonnes

---
## ETAPE 1 — Configuration PySpark + Connexion PostgreSQL

In [1]:
import os
os.environ['JAVA_HOME']             = r'C:\Program Files\Eclipse Adoptium\jdk-17.0.17.10-hotspot'
os.environ['HADOOP_HOME']           = r'C:\hadoop'
os.environ['SPARK_HOME']            = r'C:\Users\saadb\credit_scoring_env\Lib\site-packages\pyspark'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\saadb\credit_scoring_env\Scripts\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\saadb\credit_scoring_env\Scripts\python.exe'
os.environ['PATH'] = os.environ['JAVA_HOME'] + r'\bin;' + os.environ['HADOOP_HOME'] + r'\bin;' + os.environ['PATH']

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, isnull, count, round as spark_round
from pyspark.sql.functions import percentile_approx
from pyspark.sql.functions import isnull, sum as spark_sum, count
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline


JAR_PATH = "file:///C:/Users/saadb/pfe-scoring-credit/jars/postgresql-42.7.10.jar"

spark = SparkSession.builder \
    .appName("PFE_CreditScoring") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.executor.memory", "4g") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.jars", JAR_PATH) \
    .master("local[2]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f" Spark version : {spark.version}")

 Spark version : 3.5.1


---
## ETAPE 2 — Chargement du Dataset

In [2]:
JDBC_URL   = "jdbc:postgresql://localhost:5433/pfe_credit_dw"
JDBC_PROPS = {"user": "postgres", "password": "Saad2002", 
              "driver": "org.postgresql.Driver"}

df = spark.read.jdbc(url=JDBC_URL, table="marts_marts.dataset_ml_v3", 
                     properties=JDBC_PROPS)
print(f" Dataset : {df.count():,} lignes | {len(df.columns)} colonnes")

 Dataset : 8,570,101 lignes | 98 colonnes


---
## ETAPE 3 — Split Temporel

Validation temporelle — on respecte la chronologie pour eviter le data leakage :
- **Train** : 012025 a 102025 (10 mois)
- **Validation** : 112025 (1 mois) — tuning hyperparametres
- **Test** : 122025 (1 mois) — evaluation finale honnete
- **Scoring** : 012026 — prediction reelle

> Note : Decembre presente une saisonnalite (fetes) avec flag=1 = 0.038% vs 0.052% en moyenne. Ce phenomene est documente et fait partie de la realite bancaire.

In [3]:

TRAIN_PERIODS = ['012025','022025','032025','042025','052025',
                 '062025','072025','082025','092025','102025']
VAL_PERIOD    = ['112025']
TEST_PERIOD   = ['122025']
SCORE_PERIOD  = ['012026']


df_train_full = df.filter(col('is_prediction_period') == 0)
df_score      = df.filter(col('is_prediction_period') == 1)


df_train = df_train_full.filter(col('periode_trt').isin(TRAIN_PERIODS))
df_val   = df_train_full.filter(col('periode_trt').isin(VAL_PERIOD))
df_test  = df_train_full.filter(col('periode_trt').isin(TEST_PERIOD))

print(f" Train      : {df_train.count():>10,} lignes | flag=1 : {df_train.filter(col('flag_transfo')==1).count():,}")
print(f" Validation : {df_val.count():>10,} lignes | flag=1 : {df_val.filter(col('flag_transfo')==1).count():,}")
print(f" Test       : {df_test.count():>10,} lignes | flag=1 : {df_test.filter(col('flag_transfo')==1).count():,}")
print(f" Scoring    : {df_score.count():>10,} lignes ")

 Train      :  6,524,541 lignes | flag=1 : 3,411
 Validation :    675,418 lignes | flag=1 : 399
 Test       :    681,699 lignes | flag=1 : 260
 Scoring    :    688,443 lignes 


In [4]:
df.printSchema()

root
 |-- tiers_client: string (nullable = true)
 |-- periode_trt: string (nullable = true)
 |-- is_prediction_period: integer (nullable = true)
 |-- flag_transfo: integer (nullable = true)
 |-- date_trt_extr_global: timestamp (nullable = true)
 |-- age: decimal(38,18) (nullable = true)
 |-- revenu: decimal(38,18) (nullable = true)
 |-- nbr_enfant: decimal(38,18) (nullable = true)
 |-- charges: decimal(38,18) (nullable = true)
 |-- mensualite_loyer: decimal(38,18) (nullable = true)
 |-- flag_eligible_md: integer (nullable = true)
 |-- anciennete_annees: decimal(38,18) (nullable = true)
 |-- anciennete_emploi: decimal(38,18) (nullable = true)
 |-- nb_jours_dernier_evt: decimal(38,18) (nullable = true)
 |-- csp_mkt: string (nullable = true)
 |-- type_client: string (nullable = true)
 |-- groupe_civilite: string (nullable = true)
 |-- groupe_produit: string (nullable = true)
 |-- groupe_dernier_evt: string (nullable = true)
 |-- flag_canal_actif: integer (nullable = true)
 |-- groupe_rese

---
## ETAPE 4 — Exploration et Analyse des NULLs



**Strategie d'imputation :**
- NULL logique (absence produit/service) → `0`
- NULL erreur saisie → mediane training
- NULL categoriel → `INCONNU`

In [5]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost", port=5433, database="pfe_credit_dw",
    user="postgres", password="Saad2002"
)


query = """
SELECT
    col,
    ROUND(null_count * 100.0 / total, 2) AS pct_null
FROM (
    SELECT 'age' AS col, COUNT(*) FILTER (WHERE age IS NULL) AS null_count, COUNT(*) AS total FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'revenu', COUNT(*) FILTER (WHERE revenu IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nbr_enfant', COUNT(*) FILTER (WHERE nbr_enfant IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'charges', COUNT(*) FILTER (WHERE charges IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'mensualite_loyer', COUNT(*) FILTER (WHERE mensualite_loyer IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'anciennete_annees', COUNT(*) FILTER (WHERE anciennete_annees IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'anciennete_emploi', COUNT(*) FILTER (WHERE anciennete_emploi IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_jours_dernier_evt', COUNT(*) FILTER (WHERE nb_jours_dernier_evt IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_taux_credit', COUNT(*) FILTER (WHERE moy_taux_credit IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_mt_init_brut', COUNT(*) FILTER (WHERE moy_mt_init_brut IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_mt_cap_rest', COUNT(*) FILTER (WHERE moy_mt_cap_rest IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_duree_initiale', COUNT(*) FILTER (WHERE moy_duree_initiale IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_retard_echeance', COUNT(*) FILTER (WHERE moy_retard_echeance IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_credits', COUNT(*) FILTER (WHERE nb_credits IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_sav', COUNT(*) FILTER (WHERE nb_sav IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_campagnes', COUNT(*) FILTER (WHERE nb_campagnes IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_reclamations', COUNT(*) FILTER (WHERE nb_reclamations IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_demandes', COUNT(*) FILTER (WHERE nb_demandes IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'sav_moy_duree_traitement', COUNT(*) FILTER (WHERE sav_moy_duree_traitement IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'dem_moy_delai_extraction', COUNT(*) FILTER (WHERE dem_moy_delai_extraction IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
) t
ORDER BY pct_null DESC;
"""

df_null = pd.read_sql(query, conn)
conn.close()

print(df_null.to_string(index=False))

C:\Users\saadb\AppData\Local\Temp\ipykernel_7368\1077752764.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_null = pd.read_sql(query, conn)


                     col  pct_null
         nb_reclamations     99.28
dem_moy_delai_extraction     97.47
             nb_demandes     97.47
        mensualite_loyer     88.70
    nb_jours_dernier_evt     79.52
sav_moy_duree_traitement     71.52
                  nb_sav     65.61
                 charges     64.91
            nb_campagnes     44.86
              nbr_enfant     44.85
       anciennete_emploi     23.59
                  revenu     19.10
                     age     11.78
       anciennete_annees      7.83
        moy_mt_init_brut      1.58
         moy_mt_cap_rest      1.58
     moy_retard_echeance      1.58
      moy_duree_initiale      1.58
              nb_credits      1.58
         moy_taux_credit      1.58


In [6]:

conn = psycopg2.connect(
    host="localhost", port=5433, database="pfe_credit_dw",
    user="postgres", password="Saad2002"
)
query = """
SELECT col, ROUND(null_count * 100.0 / total, 2) AS pct_null
FROM (
    SELECT 'moy_montant_bien' AS col, COUNT(*) FILTER (WHERE moy_montant_bien IS NULL) AS null_count, COUNT(*) AS total FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_mt_apport', COUNT(*) FILTER (WHERE moy_mt_apport IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_mt_rachat_tot', COUNT(*) FILTER (WHERE moy_mt_rachat_tot IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_mt_vr', COUNT(*) FILTER (WHERE moy_mt_vr IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_mt_dg', COUNT(*) FILTER (WHERE moy_mt_dg IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'total_nb_impaye', COUNT(*) FILTER (WHERE total_nb_impaye IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'total_nb_impaye_regle', COUNT(*) FILTER (WHERE total_nb_impaye_regle IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'total_solde_impaye', COUNT(*) FILTER (WHERE total_solde_impaye IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'max_nb_impaye', COUNT(*) FILTER (WHERE max_nb_impaye IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_taux_credit', COUNT(*) FILTER (WHERE moy_taux_credit IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_mensualite', COUNT(*) FILTER (WHERE moy_mensualite IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_mensualite_av_der', COUNT(*) FILTER (WHERE moy_mensualite_av_der IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_duree_initiale', COUNT(*) FILTER (WHERE moy_duree_initiale IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_duree_actuelle', COUNT(*) FILTER (WHERE moy_duree_actuelle IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_nbr_ech_rest', COUNT(*) FILTER (WHERE moy_nbr_ech_rest IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_differe', COUNT(*) FILTER (WHERE moy_differe IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_impaye', COUNT(*) FILTER (WHERE flag_impaye IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_contentieux', COUNT(*) FILTER (WHERE flag_contentieux IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_credit_auto', COUNT(*) FILTER (WHERE flag_credit_auto IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_credit_equip', COUNT(*) FILTER (WHERE flag_credit_equip IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_credit_perso', COUNT(*) FILTER (WHERE flag_credit_perso IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_prel_prelevement', COUNT(*) FILTER (WHERE flag_prel_prelevement IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_credits_ctx', COUNT(*) FILTER (WHERE nb_credits_ctx IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'moy_retard_echeance', COUNT(*) FILTER (WHERE moy_retard_echeance IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'aff_moy_delai_extraction', COUNT(*) FILTER (WHERE aff_moy_delai_extraction IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
) t
ORDER BY pct_null DESC;
"""

df_null2 = pd.read_sql(query, conn)
conn.close()
print(df_null2.to_string(index=False))

C:\Users\saadb\AppData\Local\Temp\ipykernel_7368\1350977709.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_null2 = pd.read_sql(query, conn)


                     col  pct_null
        moy_nbr_ech_rest      9.15
       moy_mt_rachat_tot      1.58
               moy_mt_dg      1.58
   total_nb_impaye_regle      1.58
           max_nb_impaye      1.58
          moy_mensualite      1.58
      moy_duree_initiale      1.58
        moy_montant_bien      1.58
             flag_impaye      1.58
aff_moy_delai_extraction      1.58
     moy_retard_echeance      1.58
          nb_credits_ctx      1.58
   flag_prel_prelevement      1.58
       flag_credit_perso      1.58
       flag_credit_equip      1.58
        flag_credit_auto      1.58
        flag_contentieux      1.58
           moy_mt_apport      1.58
               moy_mt_vr      1.58
         total_nb_impaye      1.58
      total_solde_impaye      1.58
         moy_taux_credit      1.58
   moy_mensualite_av_der      1.58
      moy_duree_actuelle      1.58
             moy_differe      1.58


In [7]:
conn = psycopg2.connect(
    host="localhost", port=5433, database="pfe_credit_dw",
    user="postgres", password="Saad2002"
)

query = """
SELECT col, ROUND(null_count * 100.0 / total, 2) AS pct_null
FROM (
    SELECT 'nb_campagnes' AS col, COUNT(*) FILTER (WHERE nb_campagnes IS NULL) AS null_count, COUNT(*) AS total FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_contacts_total', COUNT(*) FILTER (WHERE nb_contacts_total IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_jours_cibles', COUNT(*) FILTER (WHERE nb_jours_cibles IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'duree_ciblage_jours', COUNT(*) FILTER (WHERE duree_ciblage_jours IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_sms_total', COUNT(*) FILTER (WHERE nb_sms_total IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_sms_failed', COUNT(*) FILTER (WHERE nb_sms_failed IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_voice_total', COUNT(*) FILTER (WHERE nb_voice_total IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_voice_failed', COUNT(*) FILTER (WHERE nb_voice_failed IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_canal_voice', COUNT(*) FILTER (WHERE flag_canal_voice IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'nb_sav', COUNT(*) FILTER (WHERE nb_sav IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'sav_nb_agences', COUNT(*) FILTER (WHERE sav_nb_agences IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'sav_nb_affaires', COUNT(*) FILTER (WHERE sav_nb_affaires IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_recouvrement', COUNT(*) FILTER (WHERE flag_recouvrement IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_main_levee', COUNT(*) FILTER (WHERE flag_main_levee IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_modification', COUNT(*) FILTER (WHERE flag_modification IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_fidelisation', COUNT(*) FILTER (WHERE flag_fidelisation IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_opposition', COUNT(*) FILTER (WHERE flag_opposition IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'sav_flag_cloture', COUNT(*) FILTER (WHERE sav_flag_cloture IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_changement_banque', COUNT(*) FILTER (WHERE flag_changement_banque IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'sav_flag_situation_credit', COUNT(*) FILTER (WHERE sav_flag_situation_credit IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_main_levee_auto', COUNT(*) FILTER (WHERE flag_main_levee_auto IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_attestation_fin', COUNT(*) FILTER (WHERE flag_attestation_fin IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'groupe_civilite', COUNT(*) FILTER (WHERE groupe_civilite IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'groupe_produit', COUNT(*) FILTER (WHERE groupe_produit IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'groupe_dernier_evt', COUNT(*) FILTER (WHERE groupe_dernier_evt IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'groupe_reseau', COUNT(*) FILTER (WHERE groupe_reseau IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'flag_canal_actif', COUNT(*) FILTER (WHERE flag_canal_actif IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'recence_sav', COUNT(*) FILTER (WHERE recence_sav IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'recence_recla', COUNT(*) FILTER (WHERE recence_recla IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL SELECT 'recence_demande', COUNT(*) FILTER (WHERE recence_demande IS NULL), COUNT(*) FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
) t
ORDER BY pct_null DESC;
"""

df_null3 = pd.read_sql(query, conn)
conn.close()
print(df_null3.to_string(index=False))

C:\Users\saadb\AppData\Local\Temp\ipykernel_7368\3579809608.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_null3 = pd.read_sql(query, conn)


                      col  pct_null
            recence_recla     99.28
          recence_demande     97.48
              recence_sav     69.18
sav_flag_situation_credit     65.61
     flag_attestation_fin     65.61
     flag_main_levee_auto     65.61
         sav_flag_cloture     65.61
                   nb_sav     65.61
          sav_nb_affaires     65.61
          flag_main_levee     65.61
        flag_fidelisation     65.61
   flag_changement_banque     65.61
           sav_nb_agences     65.61
        flag_recouvrement     65.61
        flag_modification     65.61
          flag_opposition     65.61
           nb_voice_total     62.22
          nb_voice_failed     62.22
            nb_sms_failed     45.77
             nb_sms_total     45.77
             nb_campagnes     44.86
          nb_jours_cibles     44.86
         flag_canal_voice     44.86
      duree_ciblage_jours     44.86
        nb_contacts_total     44.86
           groupe_produit      0.00
          groupe_civilite   

---
## Features Ajoutée 4 - avec calcule de taux

In [8]:

conn = psycopg2.connect(
    host="localhost", port=5433, database="pfe_credit_dw",
    user="postgres", password="Saad2002"
)
query = """
SELECT col, ROUND(null_count * 100.0 / total, 2) AS pct_null
FROM (
    SELECT 'taux_endettement' AS col, 
           COUNT(*) FILTER (WHERE taux_endettement IS NULL) AS null_count, 
           COUNT(*) AS total 
    FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL 
    SELECT 'score_risque', 
           COUNT(*) FILTER (WHERE score_risque IS NULL), 
           COUNT(*) 
    FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL 
    SELECT 'ratio_remboursement', 
           COUNT(*) FILTER (WHERE ratio_remboursement IS NULL), 
           COUNT(*) 
    FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
    UNION ALL 
    SELECT 'flag_nouveau_client', 
           COUNT(*) FILTER (WHERE flag_nouveau_client IS NULL), 
           COUNT(*) 
    FROM marts_marts.dataset_ml_v3 WHERE is_prediction_period = 0
) t
ORDER BY pct_null DESC;
"""

df_null_features = pd.read_sql(query, conn)
conn.close()
print(df_null_features.to_string(index=False))

C:\Users\saadb\AppData\Local\Temp\ipykernel_7368\3442863801.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_null_features = pd.read_sql(query, conn)


                col  pct_null
   taux_endettement     20.08
flag_nouveau_client      7.83
ratio_remboursement      1.63
       score_risque      0.00


---
## ETAPE 5 — Imputation des Valeurs Manquantes

Mediane calculee sur **training uniquement** puis appliquee sur val/test/score.

In [9]:

COLS_NULL_ZERO = [
    'moy_mt_apport','moy_mt_rachat_tot','moy_mt_vr','moy_mt_dg',
    'total_nb_impaye','total_nb_impaye_regle','total_solde_impaye',
    'max_nb_impaye','moy_mensualite','moy_mensualite_av_der',
    'flag_impaye','flag_contentieux','flag_credit_auto',
    'flag_credit_equip','flag_credit_perso','flag_prel_prelevement',
    'nb_credits_ctx','nb_credits',
    'nb_campagnes','nb_contacts_total','nb_jours_cibles',
    'duree_ciblage_jours','nb_sms_total','nb_sms_failed',
    'nb_voice_total','nb_voice_failed','flag_canal_voice',
    'flag_canal_actif',
    'nb_sav','nb_sav_non_clotures','nb_sav_sans_delai',
    'flag_recouvrement','flag_main_levee','flag_modification',
    'flag_fidelisation','flag_opposition','sav_flag_cloture',
    'flag_changement_banque','sav_flag_situation_credit',
    'flag_main_levee_auto','flag_attestation_fin',
    'sav_flag_report_echeance','flag_sav_actif','flag_sav_cloture',
    'flag_sav_annule','sav_flag_canal_tel',
    'sav_moy_duree_traitement','sav_moy_delai_extraction',
    'nb_reclamations','flag_double_prelevement','nb_recla_non_clotures',
    'nb_demandes','flag_demande_pret','dem_flag_sav','flag_simulation',
    'flag_sort_dossier','dem_flag_report_echeance','flag_rachat_credit',
    'dem_moy_delai_extraction','score_risque'
]


COLS_NULL_MEDIAN = [
    'age','revenu','nbr_enfant','charges','mensualite_loyer',
    'anciennete_annees','anciennete_emploi','nb_jours_dernier_evt',
    'moy_mt_init_brut','moy_mt_cap_rest','moy_montant_bien',
    'moy_taux_credit','moy_duree_initiale','moy_duree_actuelle',
    'moy_nbr_ech_rest','moy_differe','moy_retard_echeance',
    'aff_moy_delai_extraction',
    'taux_endettement','ratio_remboursement','flag_nouveau_client',
    'recence_sav','recence_recla','recence_demande'
]


COLS_CAT = [
    'csp_mkt', 'type_client',
    'groupe_civilite', 'groupe_produit',
    'groupe_dernier_evt', 'groupe_reseau'
]

print(f" NULL → 0       : {len(COLS_NULL_ZERO)}")
print(f" NULL → médiane : {len(COLS_NULL_MEDIAN)}")
print(f" Catégorielles  : {len(COLS_CAT)}")

 NULL → 0       : 60
 NULL → médiane : 24
 Catégorielles  : 6


In [10]:

df_train = df_train.fillna(0, subset=COLS_NULL_ZERO)
df_val   = df_val.fillna(0, subset=COLS_NULL_ZERO)
df_test  = df_test.fillna(0, subset=COLS_NULL_ZERO)
df_score = df_score.fillna(0, subset=COLS_NULL_ZERO)


In [11]:

COLS_WINSOR = [
    'revenu', 'age', 'charges', 'mensualite_loyer',
    'moy_mt_init_brut', 'moy_mt_cap_rest', 'moy_montant_bien',
    'moy_taux_credit', 'moy_mensualite', 'moy_mensualite_av_der',
    'moy_duree_initiale', 'moy_duree_actuelle', 'moy_nbr_ech_rest',
    'total_nb_impaye', 'total_solde_impaye', 'max_nb_impaye',
    'nb_jours_dernier_evt', 'anciennete_annees', 'anciennete_emploi',
    'nb_campagnes', 'nb_contacts_total', 'nb_sms_total',
    'nb_voice_total', 'nb_sav', 'nb_credits',
    'taux_endettement', 'recence_sav', 'recence_recla', 'recence_demande',
    'aff_moy_delai_extraction', 'sav_moy_duree_traitement'
]


p99_values = {}

for c in COLS_WINSOR:
    p99_val = df_train.select(
        percentile_approx(col(c), 0.99).alias('p99')
    ).collect()[0]['p99']
    p99_values[c] = float(p99_val) if p99_val is not None else None
    print(f"  {c:<35} → P99 = {p99_values[c]}")

print(f"\n P99 calculé pour {len(p99_values)} colonnes")

  revenu                              → P99 = 239395.0
  age                                 → P99 = 85.0
  charges                             → P99 = 405879.84
  mensualite_loyer                    → P99 = 81773.12
  moy_mt_init_brut                    → P99 = 355247.5
  moy_mt_cap_rest                     → P99 = 294790.76
  moy_montant_bien                    → P99 = 500678.4
  moy_taux_credit                     → P99 = 29.193512
  moy_mensualite                      → P99 = 9277.93
  moy_mensualite_av_der               → P99 = 9515.33
  moy_duree_initiale                  → P99 = 72.0
  moy_duree_actuelle                  → P99 = 72.0
  moy_nbr_ech_rest                    → P99 = 53.0
  total_nb_impaye                     → P99 = 8.0
  total_solde_impaye                  → P99 = 13012.21
  max_nb_impaye                       → P99 = 8.0
  nb_jours_dernier_evt                → P99 = 5445.0
  anciennete_annees                   → P99 = 23.0
  anciennete_emploi                   → P

In [12]:
from pyspark.sql.functions import when, col, percentile_approx



for c in COLS_WINSOR:
    if p99_values[c] is not None:
        df_train = df_train.withColumn(c,
            when(col(c) > p99_values[c], p99_values[c]).otherwise(col(c)))
        df_val = df_val.withColumn(c,
            when(col(c) > p99_values[c], p99_values[c]).otherwise(col(c)))
        df_test = df_test.withColumn(c,
            when(col(c) > p99_values[c], p99_values[c]).otherwise(col(c)))
        df_score = df_score.withColumn(c,
            when(col(c) > p99_values[c], p99_values[c]).otherwise(col(c)))

print(f" Winsorisation P99 appliquée sur {len(COLS_WINSOR)} colonnes")

 Winsorisation P99 appliquée sur 31 colonnes


In [13]:
medians = {}
for c in COLS_NULL_MEDIAN:
    median_val = df_train.select(
        percentile_approx(col(c), 0.5).alias('median')
    ).collect()[0]['median']
    medians[c] = float(median_val) if median_val is not None else 0.0
    print(f"  {c:<35} → médiane = {medians[c]}")

print(f"\n {len(medians)} médianes calculées sur training")

  age                                 → médiane = 51.0
  revenu                              → médiane = 7541.17
  nbr_enfant                          → médiane = 0.0
  charges                             → médiane = 1405.14
  mensualite_loyer                    → médiane = 2289.33
  anciennete_annees                   → médiane = 11.0
  anciennete_emploi                   → médiane = 16.0
  nb_jours_dernier_evt                → médiane = 1786.0
  moy_mt_init_brut                    → médiane = 11448.5
  moy_mt_cap_rest                     → médiane = 0.0
  moy_montant_bien                    → médiane = 11529.0
  moy_taux_credit                     → médiane = 0.002238
  moy_duree_initiale                  → médiane = 18.0
  moy_duree_actuelle                  → médiane = 16.0
  moy_nbr_ech_rest                    → médiane = 0.0
  moy_differe                         → médiane = 0.0
  moy_retard_echeance                 → médiane = 0.0
  aff_moy_delai_extraction            → médiane =

In [14]:
df_train = df_train.fillna(medians)
df_val   = df_val.fillna(medians)
df_test  = df_test.fillna(medians)
df_score = df_score.fillna(medians)


In [15]:
df_train = df_train.fillna('INCONNU', subset=COLS_CAT)
df_val   = df_val.fillna('INCONNU', subset=COLS_CAT)
df_test  = df_test.fillna('INCONNU', subset=COLS_CAT)
df_score = df_score.fillna('INCONNU', subset=COLS_CAT)
print(f' Imputation INCONNU appliquée sur {len(COLS_CAT)} colonnes catégorielles')

 Imputation INCONNU appliquée sur 6 colonnes catégorielles


In [16]:
from pyspark.sql.functions import isnull

# Vérification NULLs restants sur colonnes numériques
all_num_cols = COLS_NULL_ZERO + COLS_NULL_MEDIAN
null_remaining = {}

for c in all_num_cols:
    null_count = df_train.filter(isnull(col(c))).count()
    if null_count > 0:
        null_remaining[c] = null_count

if null_remaining:
    print(f" NULLs restants :")
    for c, n in null_remaining.items():
        print(f"  {c} → {n}")
else:
    print(' 0 NULL restant sur toutes les colonnes numériques !')

 0 NULL restant sur toutes les colonnes numériques !


---
## ETAPE 6 — One Hot Encoding

Appliquee sur val/test/score.

In [17]:

COLS_CAT = [
    'csp_mkt',            
    'type_client',        
    'groupe_civilite',    
    'groupe_produit',     
    'groupe_dernier_evt', 
    'groupe_reseau'       
]
print(f" COLS_CAT défini — {len(COLS_CAT)} colonnes catégorielles")

 COLS_CAT défini — 6 colonnes catégorielles


In [18]:

indexers = [
    StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep')
    for c in COLS_CAT
]
encoders = [
    OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_ohe', handleInvalid='keep')
    for c in COLS_CAT
]

pipeline_enc = Pipeline(stages=indexers + encoders)


encoder_model = pipeline_enc.fit(df_train)

df_train = encoder_model.transform(df_train)
df_val   = encoder_model.transform(df_val)
df_test  = encoder_model.transform(df_test)
df_score = encoder_model.transform(df_score)

print(f" Encodage OHE terminé — {len(COLS_CAT)} colonnes encodées")

 Encodage OHE terminé — 6 colonnes encodées


In [19]:



all_num_cols = COLS_NULL_ZERO + COLS_NULL_MEDIAN

null_remaining = {}
for c in all_num_cols:
    null_count = df_train.filter(isnull(col(c))).count()
    if null_count > 0:
        null_remaining[c] = null_count

if null_remaining:
    print(" NULLs restants :")
    for c, n in null_remaining.items():
        print(f"  {c} → {n}")
else:
    print(" 0 NULL restant sur toutes les colonnes numériques !")

 0 NULL restant sur toutes les colonnes numériques !


In [20]:

COLS_TO_DROP = [
    
    'date_trt_extr_global',
    
    'aff_moy_delai_extraction',
    'sav_moy_delai_extraction',
    'dem_moy_delai_extraction',
   
] + COLS_CAT \
  + [c + '_idx' for c in COLS_CAT]

df_train_clean = df_train.drop(*COLS_TO_DROP)
df_val_clean   = df_val.drop(*COLS_TO_DROP)
df_test_clean  = df_test.drop(*COLS_TO_DROP)
df_score_clean = df_score.drop(*COLS_TO_DROP)

print(f" Colonnes avant suppression : {len(df_train.columns)}")
print(f" Colonnes supprimées        : {len(COLS_TO_DROP)}")
print(f" Colonnes finales           : {len(df_train_clean.columns)}")

 Colonnes avant suppression : 110
 Colonnes supprimées        : 16
 Colonnes finales           : 94


In [21]:
print(f"Total colonnes : {len(df_train_clean.columns)}")
print("\nToutes les colonnes :")
for i, c in enumerate(df_train_clean.columns):
    print(f"  {i+1:>3}. {c}")

Total colonnes : 94

Toutes les colonnes :
    1. tiers_client
    2. periode_trt
    3. is_prediction_period
    4. flag_transfo
    5. age
    6. revenu
    7. nbr_enfant
    8. charges
    9. mensualite_loyer
   10. flag_eligible_md
   11. anciennete_annees
   12. anciennete_emploi
   13. nb_jours_dernier_evt
   14. flag_canal_actif
   15. nb_credits
   16. moy_mt_init_brut
   17. moy_mt_cap_rest
   18. moy_montant_bien
   19. moy_mt_apport
   20. moy_mt_rachat_tot
   21. moy_mt_vr
   22. moy_mt_dg
   23. total_nb_impaye
   24. total_nb_impaye_regle
   25. total_solde_impaye
   26. max_nb_impaye
   27. moy_taux_credit
   28. moy_mensualite
   29. moy_mensualite_av_der
   30. moy_duree_initiale
   31. moy_duree_actuelle
   32. moy_nbr_ech_rest
   33. moy_differe
   34. flag_impaye
   35. flag_contentieux
   36. flag_credit_auto
   37. flag_credit_equip
   38. flag_credit_perso
   39. flag_prel_prelevement
   40. nb_credits_ctx
   41. moy_retard_echeance
   42. nb_campagnes
   43. nb_

---
# ETAPE 7 — VectorAssembler



In [22]:
from pyspark.ml.feature import VectorAssembler


COLS_EXCLUDE = ['tiers_client', 'periode_trt',
                'is_prediction_period', 'flag_transfo']


feature_cols = [c for c in df_train_clean.columns
                if c not in COLS_EXCLUDE]

print(f" Nombre de features : {len(feature_cols)}")


assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features',
    handleInvalid='skip'
)

df_train_assembled = assembler.transform(df_train_clean)
df_val_assembled   = assembler.transform(df_val_clean)
df_test_assembled  = assembler.transform(df_test_clean)
df_score_assembled = assembler.transform(df_score_clean)

print(f" VectorAssembler appliqué sur train/val/test/score")
print(f" Colonnes finales : {len(df_train_assembled.columns)}")

 Nombre de features : 90
 VectorAssembler appliqué sur train/val/test/score
 Colonnes finales : 95


In [23]:
from pyspark.ml.feature import VectorAssembler


COLS_EXCLUDE = ['tiers_client', 'periode_trt',
                'is_prediction_period', 'flag_transfo']


feature_cols = [c for c in df_train_clean.columns
                if c not in COLS_EXCLUDE]

print(f" Nombre de features : {len(feature_cols)}")


assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features',
    handleInvalid='skip'
)

df_train_assembled = assembler.transform(df_train_clean)
df_val_assembled   = assembler.transform(df_val_clean)
df_test_assembled  = assembler.transform(df_test_clean)
df_score_assembled = assembler.transform(df_score_clean)

print(f" VectorAssembler appliqué sur train/val/test/score")
print(f" Colonnes finales : {len(df_train_assembled.columns)}")

 Nombre de features : 90
 VectorAssembler appliqué sur train/val/test/score
 Colonnes finales : 95


In [24]:


conn = psycopg2.connect(
    host="localhost", port=5433, database="pfe_credit_dw",
    user="postgres", password="Saad2002"
)

query = """
SELECT 
    COUNT(*) AS total,
    SUM(flag_transfo) AS flag1,
    COUNT(*) - SUM(flag_transfo) AS flag0,
    ROUND(SUM(flag_transfo)*100.0/COUNT(*), 4) AS pct_flag1
FROM marts_marts.dataset_ml_v3
WHERE periode_trt IN ('012025','022025','032025','042025','052025',
                      '062025','072025','082025','092025','102025');
"""

df_balance = pd.read_sql(query, conn)
conn.close()
print(df_balance.to_string(index=False))
print(f"\nRatio déséquilibre : 1 flag=1 pour {int(df_balance['flag0'][0])//int(df_balance['flag1'][0])} flag=0")

C:\Users\saadb\AppData\Local\Temp\ipykernel_7368\4071686473.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_balance = pd.read_sql(query, conn)


  total  flag1   flag0  pct_flag1
6524541   3411 6521130     0.0523

Ratio déséquilibre : 1 flag=1 pour 1911 flag=0


In [25]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost", port=5433, database="pfe_credit_dw",
    user="postgres", password="Saad2002"
)


query = """
SELECT *
FROM marts_marts.dataset_ml_v3
WHERE periode_trt IN ('012025','022025','032025','042025','052025',
                      '062025','072025','082025','092025','102025')
AND flag_transfo = 1
"""

df_flag1 = pd.read_sql(query, conn)
conn.close()

print(f" flag=1 chargé : {df_flag1.shape}")

C:\Users\saadb\AppData\Local\Temp\ipykernel_7368\2346374863.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_flag1 = pd.read_sql(query, conn)


 flag=1 chargé : (3411, 98)
